In [1]:
import numpy as np
import json
import re
from konlpy.tag import Okt
from kiwipiepy import Kiwi
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
from wordcloud import WordCloud
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import matplotlib
import warnings
warnings.filterwarnings(action='ignore')
import folium
import seaborn as sns
import requests
from bs4 import BeautifulSoup
import time
from googlesearch import search

In [2]:
import re
from collections import Counter
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud
font_path = "c:/Windows/fonts/malgun.ttf"
font_name = font_manager.FontProperties( fname = font_path).get_name()
matplotlib.rc('font', family = font_name)

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [4]:
main_df = pd.read_csv('./midterm/group_technique_n.csv')

In [5]:
target_tactics = ['collection', 'exfiltration', 'impact']

# 1. 3대 핵심 텍틱에 해당하는 행들만 전체 데이터에서 필터링
condition = main_df['tactics'].fillna('').str.lower().apply(
    lambda x: any(tactic in x for tactic in target_tactics)
)
final_stage_df = main_df[condition]
print(final_stage_df['technique_id'].value_counts().head(10))

technique_id
T1560    61
T1005    45
T1074    44
T1056    34
T1114    30
T1567    29
T1041    27
T1119    21
T1213    19
T1113    19
Name: count, dtype: int64


In [7]:
target_tactics = ['collection', 'exfiltration', 'impact']
target_countries = ['China', 'Russia', 'Iran', 'North Korea']

# 1. 대상 4개 국가만 먼저 필터링
country_filtered_df = main_df[main_df['Final_Country'].isin(target_countries)]

# 2. 수집/유출/파괴 전술이 포함된 행을 찾기 위한 조건 설정
condition = country_filtered_df['tactics'].fillna('').str.lower().apply(
    lambda x: any(tactic in x for tactic in target_tactics)
)

raw_final_stage_df = country_filtered_df[condition].copy()

raw_final_stage_df = raw_final_stage_df.sort_values(by='Final_Country')


In [8]:
final_df = raw_final_stage_df

In [9]:
final_df[final_df['Final_Country'] == 'China']['technique_id'].value_counts()

technique_id
T1560    31
T1074    24
T1005    19
T1056    13
T1041    10
T1114     9
T1567     9
T1048     8
T1119     8
T1113     6
T1213     5
T1657     5
T1039     4
T1486     4
T1030     3
T1496     2
T1537     2
T1530     2
T1557     2
T1020     2
T1052     2
T1485     1
T1602     1
T1125     1
T1490     1
T1667     1
Name: count, dtype: int64

In [10]:
final_df[final_df['Final_Country'] == 'Iran']['technique_id'].value_counts()

technique_id
T1074    9
T1560    8
T1567    8
T1114    8
T1005    7
T1041    7
T1056    6
T1113    5
T1119    4
T1048    4
T1213    4
T1486    4
T1657    3
T1490    3
T1115    2
T1530    2
T1489    2
T1561    2
T1039    1
T1125    1
T1557    1
T1485    1
T1123    1
T1025    1
Name: count, dtype: int64

In [11]:
final_df[final_df['Final_Country'] == 'Russia']['technique_id'].value_counts()

technique_id
T1005    12
T1560    11
T1114    10
T1213     9
T1567     7
T1074     7
T1119     6
T1113     6
T1486     5
T1056     5
T1041     4
T1491     4
T1657     3
T1048     3
T1039     3
T1561     3
T1025     3
T1020     3
T1557     2
T1485     2
T1490     2
T1125     2
T1489     2
T1531     1
T1030     1
T1537     1
T1565     1
T1498     1
T1499     1
Name: count, dtype: int64

In [12]:
final_df[final_df['Final_Country'] == 'North Korea']['technique_id'].value_counts()

technique_id
T1560    9
T1056    8
T1005    6
T1486    5
T1567    4
T1041    4
T1529    4
T1561    4
T1657    4
T1565    3
T1557    3
T1114    3
T1489    3
T1496    2
T1074    2
T1115    2
T1113    2
T1485    2
T1530    1
T1123    1
T1020    1
T1531    1
T1030    1
T1048    1
T1213    1
T1490    1
T1185    1
T1491    1
T1029    1
Name: count, dtype: int64

중국 핵심 단어: exfiltration (56회), microsoft (33회), fin (33회), mandiant (30회), exfiltrated (30회), archive (26회)
북한 핵심 단어: kimsuky (49회), ransomware (41회), medusa (39회), novetta (22회), blockbuster (22회), lazarus (19회)
러시아 핵심 단어: gru (24회), gamaredon (24회), eset (21회), sandworm (14회)
이란 핵심 단어: void (31회), manticore (31회), march (31회), handala (26회), hack (26회), fbi (21회), beavertail (15회)

1. 🇨🇳 중국 대응: "한국형 셰어드 인텔리전스(Shared Intelligence) 및 핵심 기술 금고화"
중국 키워드에서 exfiltration(대량 유출)과 archive(압축)가 압도적이었다는 것은, 그들이 국내 기업이나 연구소에 침투해 정보를 빼갈 때 우리가 유출 자체를 눈치채지 못하고 있었다는 뜻입니다.

보완 방안:

제로 트러스트 기반 데이터 보호(DLP) 강화: 중요 기밀 데이터(반도체, 방산, 바이오 등)가 보관된 서버는 외부 인터넷과 완전히 격리하고, 대용량 파일이 압축되거나 외부로 나가는 통로를 원천 차단하는 시스템을 의무화해야 합니다.

MS 인프라 취약점 공동 대응 시스템: microsoft가 핵심 키워드인 만큼, 국내 정부 기관 및 기업들이 사용하는 MS 가상화 환경이나 메일 서버의 취약점(1-Day) 패치를 강제하고 실시간으로 공유하는 민관 합동 관제 체계를 고도화해야 합니다.

2. 🇰🇵 북한 대응: "랜섬웨어 전담 대응 조직 운영 및 킴수키 시그니처 차단"
북한의 ransomware(랜섬웨어), kimsuky(킴수키) 키워드는 한국의 공공기관과 핵심 인프라를 상대로 한 실질적인 위협과 금전적 타격을 의미합니다.

보완 방안:

공공·금융 분야 백업 체계의 전면 개편: 북한 랜섬웨어에 감염되더라도 데이터 파괴(impact) 피해를 최소화할 수 있도록, 네트워크와 물리적으로 완전히 분리된 '오프라인 백업(Air-Gapped Backup)'을 의무화해야 합니다.

킴수키(Kimsuky) 전용 스피어피싱 방어망 구축: 킴수키의 주 무기는 한국 임직원을 노리는 교묘한 이메일 피싱입니다. 악성 첨부파일 문서(file, macro) 내의 코드를 자동으로 가상 공간에서 실행해 검사하는 EDR(단말 탐지) 및 이메일 격리 기술을 전면 도입해야 합니다.

3. 🇷🇺 러시아 대응: "국가 핵심 기반시설(OT)의 전시(戰時) 방어 시나리오 구축"
러시아 군 정보부(gru)와 샌드웜(sandworm)이 등장했다는 것은, 이들이 민간 해커 수준이 아니라 국가의 명령을 받고 상대국의 발전소, 가스망, 철도 등 기반시설을 마비시키는 파괴 공작에 특화되어 있다는 뜻입니다.

보완 방안:

폐쇄망(OT 영역)의 실시간 모니터링 체계 도입: 일반 IT 망과 분리되어 안전하다고 믿는 발전소나 국방 네트워크 안에서도 내부 정찰과 파괴 행위가 일어날 수 있음을 인지해야 합니다. 폐쇄망 전용 이상 트래픽 탐지 시스템을 도입해야 합니다.

우크라이나 전장 기반의 위협 인텔리전스(CTI) 흡수: 러시아의 gamaredon 등이 우크라이나에서 썼던 최신 사이버 무기 패턴을 글로벌 보안 기업(eset, microsoft)과의 정보 동맹을 통해 실시간으로 인도받아 국내 방어망에 선제 반영해야 합니다.

4. 🇮🇷 이란 대응: "지정학적 리스크에 따른 사이버 보복 공급망(Supply Chain) 점검"
이란의 void manticore, handala 같은 해킹 그룹은 지정학적 갈등(이스라엘-이란 등)이 발생했을 때 상대국을 향해 무차별 파괴 및 심리전 해킹(hack)을 감행합니다. 한국 역시 중동 정세나 동맹국 관계에 따라 언제든 보복 타깃이 될 수 있습니다.

보완 방안:

공급망 보안(Supply Chain Security) 강화: 이란 계열 해커들은 보안이 삼엄한 정부 기관을 직접 찌르기보다, 정부에 소프트웨어를 납품하는 중소 협력업체를 먼저 뚫고 들어오는 방식을 즐겨 씁니다. 공급망 전체의 보안 수준을 함께 끌어올리는 인증 제도가 필요합니다.

핵심 인물 및 기관에 대한 사이버 억제력 확보: fbi 등 국제 정보 기관과의 공조를 통해 위협 해커들의 신원과 인프라를 추적하고, 국제 사회와 함께 해킹 거점을 무력화하는 능동적 사이버 방어(Active Cyber Defense) 체계로 전환해야 합니다.

가중치 넣을 수 있는 

나라 가중치 1개

누나꺼 결과 그거 코드들 나라별로 가중치 다르게 하고

내꺼 공격 성공한 코드들 나라별로 가중치 다르게 해서
